In [1]:
import string
import nltk
nltk.download('stopwords')
nltk.download('punkt')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
DATASET_PATH = '/content/2cls_spam_text_cls.csv'
df = pd.read_csv(DATASET_PATH)
df

FileNotFoundError: [Errno 2] No such file or directory: '/content/2cls_spam_text_cls.csv'

In [ ]:
messages = df['Message'].values.tolist()
labels = df['Category'].values.tolist()

In [ ]:
def lowercase(input):
  return input.lower()

In [ ]:
def punctuation_removal(input):
  translator = str.maketrans('', '', string.punctuation)
  return input.translate(translator)

In [ ]:
def tokenize(input):
  return nltk.word_tokenize(input)

In [ ]:
def remove_stopwords(input):
  stop_words = nltk.corpus.stopwords.words('english')
  return [token for token in input if token not in stop_words]

In [ ]:
def stemming(input):
  stemmer = nltk.PorterStemmer()
  return [stemmer.stem(token) for token in input]

In [ ]:
def preprocess_text(input):
  input = lowercase(input)
  input = punctuation_removal(input)
  input = tokenize(input)
  input = remove_stopwords(input)
  input = stemming(input)
  return input


In [ ]:
messages = [preprocess_text(message) for message in messages]

In [ ]:
messages[0]

In [ ]:
def create_dictionary(messages):
  dictionary = []
  for message in messages:
    for token in message:
      if token not in dictionary:
        dictionary.append(token)
  return dictionary


In [ ]:
def create_features(tokens):
  features = np.zeros(len(dictionary))
  for token in tokens:
    if token in dictionary:
      features[dictionary.index(token)] += 1
  return features


In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels)
print(f'Classes: {le.classes_}')
print(f'Encoded labels; {y}')

In [ ]:
dictionary = create_dictionary(messages)
x = np.array([create_features(token) for token in messages])

In [ ]:
VAL_SIZE = 0.2
TEST_SIZE = 0.125
SEED = 0
IS_SHUFFLE = True

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=VAL_SIZE, shuffle=IS_SHUFFLE, random_state=SEED)
x_train, x_test, y_train, y_test = train_test_split(x_train, y_train, test_size=TEST_SIZE, shuffle=IS_SHUFFLE, random_state=SEED)

In [ ]:
model = GaussianNB()
model = model.fit(x_train, y_train)

In [ ]:
y_val_pred = model.predict(x_val)
y_test_pred = model.predict(x_test)
val_accuracy = accuracy_score(y_val, y_val_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)